In [ ]:
# pandas, numpy, seaborn, matplotlib, scikit-learn
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# sklearn preprocessing tools: StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
#  sklearn imputer: SimpleImputer
from sklearn.impute import SimpleImputer

# Display settings
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 100)

In [ ]:
# === LOAD THE DATA ===
# Load Titanic dataset from seaborn
# Save as titanic_raw - this is our Original backup copy of the data
titanic_raw = sns.load_dataset('titanic')

In [ ]:
# === FIRST LOOK AT THE DATA ===
# Display the first 10 rows of the dataset
titanic_raw.head(10)

In [ ]:
# === ESSENTIAL WORKFLOW ===
#  shape - how big is the data?
print("Shape of the dataset:", titanic_raw.shape)

In [ ]:
# info() - what are the data types and non-null counts (what's missing?)
print(titanic_raw.info())


In [ ]:
# describe - what are the summary statistics for numeric columns?
print(titanic_raw.describe())

In [ ]:
# === STEP 1: CHECK FOR DUPLICATES ===
#  create working copy: df = titanic_raw.copy()
# count duplicates : df.duplicated().sum()
df = titanic_raw.copy()
print(f"Duplicate rows: {df.duplicated().sum()}")

In [ ]:
# EXAMPLE : what duplicates look like
duplicates_df = df[df.duplicated(keep=False)]
duplicates_df

In [ ]:
duplicates_df.sort_values(['pclass','who']).head(20)

In [ ]:
# === REMOVE DUPLICATES ===
#  Use drop_duplicates() to remove duplicates
# Show before/after row counts
print(f"Rows before removing duplicates: {df.shape}")
df.drop_duplicates()
print(f"Rows after removing duplicates: {df.shape}")

In [ ]:
# === DUPLICATES ON SPECIFIC COLUMNS ===
# Check for duplicates based on 'pclass' and 'who'
duplicate_subset = df.duplicated(subset=['pclass', 'who'])
# Keep first or last occurrence
print(f"Duplicate rows based on 'pclass' and 'who': {duplicate_subset.sum()}")



In [ ]:
# === STEP 2: MISSING VALUES ANALYSIS ===
# Count missing values: df.isnull().sum()
# Calculate missing percentage: (df.isnull().sum() / len(df)) * 100

missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame(
    {
        'Missing Count': missing_counts,
        'Missing Percentage': missing_pct
    }
)
print(missing_df)

missing_df[missing_df['Missing Count'] > 0]

In [ ]:
# === STEP 2a: DROP SPARSE COLUMNS ===
# Drop columns with more than 50% missing values
# Drop 'deck' column (77% missing)
print(f"Number of columns before dropping sparse columns: {len(df.columns)}")
df = df.drop(columns=['deck'])
print(f"Number of columns after dropping sparse columns: {len(df.columns)}")

# SimpleImputer: The Professional Approach to Handling Missing Data
WHY USE SimpleImputer INSTEAD OF fillna()?

fillna() is great for quick exploratory work.
SimpleImputer is designed for production-ready data preprocessing because :

1. CONSISTENCY: SimpleImputer ensures the same imputation strategy is applied across all datasets (train, test, new data) by fitting on the training data and transforming consistently. It remembers the imputation values, preventing data leakage and ensuring reproducibility.

2. PIPELINE INTEGRATION: SimpleImputer can be seamlessly integrated into scikit-learn pipelines, allowing for streamlined preprocessing and model training in a single workflow.

3. options: SimpleImputer offers various imputation strategies (mean, median, most_frequent, constant) and can handle both numerical and categorical data effectively.

In [ ]:
# === STEP 2b: FILL AGE WITH SIMPLEIMPUTER ===
# Create SimpleImputer with strategy='median' for 'age' column
age_imputer = SimpleImputer(strategy='median')
# Fit and transform the 'age' column
df['age'] = age_imputer.fit_transform(df[['age']])
# CHECK IMPUTER.STATISTICS_ - this shows the median value used for imputation
print(f"Median age used for imputation: {age_imputer.statistics_[0]}")
# Verify that there are no more missing values in 'age'
print(f"Missing values in 'age' after imputation: {df['age'].isnull().sum()}")

